# Pursuit-evasion: contorno di $V_0=0$ al primo campione negativo

Riproduce la figura RSS usando `scripts/pursuit_th.py` e la simulazione già salvata, poi aggiunge il contorno terminale attorno alla posa sbiadita dell'ego. Nessuna nuova simulazione e nessuna modifica ai file Python esistenti.

Eseguire con il kernel del progetto dopo aver salvato questo notebook in `grafici/pursuit_notebook.ipynb`. Servono `scripts/pursuit_rss_neg/simulation.npz` e il corrispondente `results/brt/rss.npz` originale (i BRT non sono inclusi su GitHub).


In [11]:
from pathlib import Path
import importlib.util
import json
import sys
from unittest.mock import patch

import numpy as np
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "pursuit_th.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla cartella grafici del repository.")

scripts = ROOT / "scripts"
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))
spec = importlib.util.spec_from_file_location("pursuit_notebook_source", scripts / "pursuit_th.py")
pursuit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pursuit)
# L'importazione non esegue main() e non genera file.


In [12]:
SIMULATION_FILE = ROOT / "scripts" / "pursuit_rss_neg" / "simulation.npz"
V0_FILE = ROOT / "results" / "brt" / "rss.npz"
OUTPUT_FOLDER = ROOT / "grafici" / "output" / "pursuit_rss_neg"
OUTPUT_NAME = "trajectory_v0.pdf"

CONTOUR_COLOR = "darkred"
CONTOUR_LINEWIDTH = 1.8
CONTOUR_LINESTYLE = "--"
CONTOUR_ALPHA = 0.8
CONTOUR_LABEL = r"$V_0=0$"
NX, NY = 501, 401
# False mantiene i limiti della figura originale; True include tutto il contorno.
EXPAND_LIMITS = False


In [13]:
for path in (SIMULATION_FILE, V0_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"File richiesto: {path}")

with np.load(SIMULATION_FILE, allow_pickle=False) as data:
    result = {name: data[name] for name in ("time", "states", "v0", "brt", "recovery")}
    config = json.loads(str(data["configuration_json"].item()))
metric = config["metric"]
for name, key in (("EGO_LENGTH", "ego_length"), ("EGO_WIDTH", "ego_width"),
                  ("HUMAN_LENGTH", "human_length"), ("HUMAN_WIDTH", "human_width")):
    if key in config and not np.isclose(config[key], getattr(pursuit, name)):
        raise ValueError(f"Geometria salvata differente: {key}")

negative = np.flatnonzero(np.isfinite(result["v0"]) & (result["v0"] < 0))
if negative.size == 0:
    raise ValueError("La simulazione non contiene campioni finiti con V0 < 0.")
k = int(negative[0])
z = result["states"][k]

# Carica solo il campo terminale, evitando BRT e gradienti 6D non necessari.
with np.load(V0_FILE, allow_pickle=False) as data:
    axes = tuple(np.asarray(data[name], dtype=float) for name in pursuit.NAMES)
    v0 = data["V0"]
    lo, hi = np.asarray(data["grid_lo"], float), np.asarray(data["grid_hi"], float)
    periodic = tuple(int(i) for i in data["periodic_dims"])
    if "state_names" in data and tuple(data["state_names"].tolist()) != pursuit.NAMES:
        raise ValueError("Ordine degli stati incompatibile.")
if v0.shape != tuple(len(axis) for axis in axes) or periodic != (2,):
    raise ValueError("Griglia incompatibile con pursuit_th.py.")
for dim, axis in enumerate(axes):
    if (axis.ndim != 1 or len(axis) < 2 or not np.isfinite(axis).all()
            or not np.all(np.diff(axis) > 0) or not np.isclose(axis[0], lo[dim])):
        raise ValueError(f"Asse non valido: {pursuit.NAMES[dim]}")
    if dim == 2:
        if not np.isclose(hi[dim] - lo[dim], 2 * np.pi) or axis[-1] >= hi[dim]:
            raise ValueError("Asse angolare periodico incompatibile.")
    elif not np.isclose(axis[-1], hi[dim]):
        raise ValueError(f"Limite superiore incompatibile: {pursuit.NAMES[dim]}")

# Adattatore per riusare interpolate(): qui anche il primo campo è V0;
# il gradiente nullo è una vista broadcast, senza allocazione di un array 6D.
model = dict(axes=axes, lo=lo, hi=hi, v0=v0, brt=v0,
             gradient=np.broadcast_to(np.zeros(6), v0.shape + (6,)))
state = pursuit.relative_state(z, model)
if not pursuit.inside_grid(state, model):
    raise ValueError("Il campione selezionato è fuori dalla griglia.")
_, terminal, _ = pursuit.interpolate(model, state)
saved_terminal = float(result["v0"][k])
if not np.isfinite(terminal) or terminal >= 0 or not np.isclose(
        terminal, saved_terminal, rtol=1e-4, atol=1e-6):
    raise ValueError(f"V0 non corrispondente alla simulazione: salvato={saved_terminal}, "
                     f"ricalcolato={terminal}. Selezionare il file NPZ originale.")
print(f"Primo V0 < 0: campione {k}, t={result['time'][k]:.6f} s")
print(f"V0 salvato={saved_terminal:.8g}; ricalcolato={terminal:.8g}")
print(dict(zip(pursuit.NAMES, state)))


Primo V0 < 0: campione 42, t=1.400000 s
V0 salvato=-0.0070061109; ricalcolato=-0.0070061109
{'x_rel': np.float64(-8.426189954270642), 'y_rel': np.float64(2.2702935521154073), 'theta_rel': np.float64(0.03994275869194963), 'v_H': np.float64(7.283333333333325), 'delta_E': np.float64(-0.1218), 'v_E': np.float64(7.966666666666668)}


La fetta mantiene i valori di $\theta_{rel},v_H,\delta_E,v_E$ del campione selezionato, interpolando fra i nodi con la funzione originale, compresa la periodicità angolare. Si varia soltanto $(x_{rel},y_{rel})$. Il contorno viene ruotato dell'heading dell'ego e traslato nella sua posizione globale. L'appartenenza a $V_0<0$ riguarda lo stato relativo (il centro di riferimento dell'human), senza implicare una collisione geometrica fra le sagome.


In [14]:
slice_v0 = np.empty((len(axes[0]), len(axes[1])), dtype=float)
query = state.copy()
for i, x in enumerate(axes[0]):
    for j, y in enumerate(axes[1]):
        query[:2] = x, y
        _, slice_v0[i, j], _ = pursuit.interpolate(model, query)
if not np.isfinite(slice_v0).all() or not slice_v0.min() < 0 < slice_v0.max():
    raise ValueError("La fetta non contiene un contorno zero valido.")

# Raffina solo la visualizzazione bilineare: nessuno smoothing del campo.
slice_interpolator = RegularGridInterpolator(axes[:2], slice_v0, bounds_error=True)
assert np.isclose(float(slice_interpolator(state[None, :2])[0]), terminal,
                  rtol=1e-6, atol=1e-7)
x = np.linspace(axes[0][0], axes[0][-1], NX)
y = np.linspace(axes[1][0], axes[1][-1], NY)
XR, YR = np.meshgrid(x, y, indexing="ij")
V0_slice = slice_interpolator(np.stack((XR, YR), axis=-1))
c, s = np.cos(z[2]), np.sin(z[2])
XG = z[0] + c * XR - s * YR
YG = z[1] + s * XR + c * YR
assert np.allclose([z[0] + c * state[0] - s * state[1],
                    z[1] + s * state[0] + c * state[1]], z[5:7])
edge_negative = np.any(np.concatenate((slice_v0[0], slice_v0[-1],
                                       slice_v0[:, 0], slice_v0[:, -1])) < 0)
if edge_negative:
    print("La regione negativa raggiunge il bordo della griglia: il contorno può essere aperto.")


La regione negativa raggiunge il bordo della griglia: il contorno può essere aperto.


In [15]:
pursuit.configure_style()
captured = []
def capture_figure(fig, folder, name):
    captured.append(fig)

# Intercetta soltanto il salvataggio durante questa chiamata: la funzione
# originale disegna traiettorie, sagome sbiadite, stelle e colorbar.
with patch.object(pursuit, "save", side_effect=capture_figure):
    pursuit.plot_trajectory(result, metric, OUTPUT_FOLDER)
fig = captured[0]
ax = fig.axes[0]
contour = ax.contour(XG, YG, V0_slice, levels=[0], colors=[CONTOUR_COLOR],
                    linewidths=CONTOUR_LINEWIDTH, linestyles=CONTOUR_LINESTYLE,
                    alpha=CONTOUR_ALPHA, zorder=7)
handles, labels = list(ax.get_legend().legend_handles), [
    text.get_text() for text in ax.get_legend().get_texts()]
handles.append(pursuit.Line2D([], [], color=CONTOUR_COLOR,
    linewidth=CONTOUR_LINEWIDTH, linestyle=CONTOUR_LINESTYLE, alpha=CONTOUR_ALPHA))
labels.append(CONTOUR_LABEL)
ax.legend(handles=handles, labels=labels, loc="lower left")

if EXPAND_LIMITS:
    vertices = [segment for segment in contour.allsegs[0] if len(segment)]
    if vertices:
        points = np.concatenate(vertices)
        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        pad = 0.5
        ax.set_xlim(min(xlim[0], points[:, 0].min() - pad),
                    max(xlim[1], points[:, 0].max() + pad))
        ax.set_ylim(min(ylim[0], points[:, 1].min() - pad),
                    max(ylim[1], points[:, 1].max() + pad))

pursuit.format_colorbars(fig)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_FOLDER / OUTPUT_NAME
fig.savefig(output_file, dpi=pursuit.DPI, bbox_inches="tight", pad_inches=0.15)
display(fig)
pursuit.plt.close(fig)
print(f"Figura salvata: {output_file}")


'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp


<Figure size 920x600 with 2 Axes>

Figura salvata: /home/alessio-grisorio/TESI/hj_reachability/grafici/output/pursuit_rss_neg/trajectory_v0.pdf
